# SPILLWAY1.0.1 / Qwen standard 949 attack +97 benign / Root reviewed8e003
이 링크의 새 Root gate가 포함되어 있습니다. GPU 유형을 RTX PRO6000 Blackwell(실제 ≥90000MiB), Python3.13으로 선택한 뒤 위에서부터 실행하세요. 첫 셀은 소스·gate SHA를 확인하며, 두 번째 셀에서 조건을 모두 확인한 후 모델 준비와 실험을 시작합니다.

이번 변경은 judge 준비 단계입니다. 비어 있지 않은 중립 warm-up 세 번 모두 양수 생성 토큰이 있어야 하며, 마지막 준비 응답은 5초 미만이어야 합니다. 원 평가 judge timeout30초, SDK f93d, 모델·prompt·decode·context·scorer·payload는 유지합니다. CPU 검산은 실제 Colab/GPU 성공을 뜻하지 않습니다.

이 패킷은 1046개 user UID와 별도 standalone 35개만 다룹니다. PISmith Qwen은 실행하지 않습니다. standard와 ChatInject는 별도 VM에서 실행하세요. 같은 VM을 재사용하면 이전 실행이 끝난 뒤 런타임 > 세션 다시 시작을 먼저 누르세요. 진행 중인 Coder004 VM은 건드리지 마세요.

결과는 이 노트북 전용 Drive 폴더에 저장됩니다. 실패한 이전 폴더는 보존하고 새 RUN_ID를 사용합니다. 자동 재실행은 없습니다.


In [ ]:
import hashlib, json, pathlib, sys, zipfile, urllib.request
assert sys.version_info[:2] == (3,13), "Original Python3.13 family required; no fallback"
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/SPILLWAY_101_QWEN_STANDARD_20261117_8E003')
ARTIFACT_BASE = 'https://raw.githubusercontent.com/saintmachines/spillway/2023c9231a4ef8c775f08a7aef91bb67823ba6a0/reproduction/final101-qwen8e003/'
ARCHIVE = pathlib.Path('/content') / 'SPILLWAY_101_A_QWEN_STANDARD_PACKET_8E003.zip'
if not ARCHIVE.is_file() or hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() != 'bb5aa0d783b9d8f3bb4ccdd3723df702150cfe9fa7d4972a5c30d53103e8f965':
    with urllib.request.urlopen(ARTIFACT_BASE + 'SPILLWAY_101_A_QWEN_STANDARD_PACKET_8E003.zip', timeout=120) as response:
        packet_bytes = response.read()
    assert hashlib.sha256(packet_bytes).hexdigest() == 'bb5aa0d783b9d8f3bb4ccdd3723df702150cfe9fa7d4972a5c30d53103e8f965'
    ARCHIVE.write_bytes(packet_bytes)
GATE_LOCAL_PATH = pathlib.Path('/content') / 'ROOT_EXECUTION_GATE_colab_qwen_8e003.json'
with urllib.request.urlopen(ARTIFACT_BASE + 'ROOT_EXECUTION_GATE_colab_qwen_8e003.json', timeout=60) as response:
    gate_bytes = response.read()
assert hashlib.sha256(gate_bytes).hexdigest() == '9f6f12a742bdf1255409af6cb2a2b327ca3644594ee1960ecef9925a9ecc261c'
GATE_LOCAL_PATH.write_bytes(gate_bytes)
assert hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() == 'bb5aa0d783b9d8f3bb4ccdd3723df702150cfe9fa7d4972a5c30d53103e8f965'
ROOT = pathlib.Path('/content') / 'spw101_colab_qwen_8e003_bb5aa0d783b9'
if not ROOT.exists():
    with zipfile.ZipFile(ARCHIVE) as z:
        names = z.namelist()
        assert len(names) == len(set(names))
        assert sum(i.file_size for i in z.infolist()) <= 200_000_000
        assert all(not pathlib.PurePosixPath(n).is_absolute() and '..' not in pathlib.PurePosixPath(n).parts for n in names)
        ROOT.mkdir()
        z.extractall(ROOT)
for module_name in ('aggregate', 'attack_adapters', 'closure', 'common', 'dependency_lock', 'evaluation_environment', 'measurement_worker_entry', 'model_server', 'native_auxiliary', 'paper_scoring', 'preprovider_accounting', 'progress_observer', 'provider_audit', 'qwen101_controller', 'qwen101_lane', 'runtime', 'scorer_guard', 'worker'):
    loaded = sys.modules.get(module_name)
    if loaded is not None:
        assert pathlib.Path(loaded.__file__).resolve().parent == ROOT.resolve(), (
            'A different packet is cached: wait for the old run to finish and restart the Python session.')
sys.path.insert(0, str(ROOT))
import runtime
print(json.dumps(runtime.inspect_packet(ROOT), indent=2))


In [ ]:
# Run only in a free Colab VM. The controller checks source, gate, Python, GPU, ownership and model hashes.
from qwen101_controller import run
ROOT_GATE = GATE_LOCAL_PATH
RUN_ID = 'final101_standard_colab_qwen_20261117_8e003'
result = run(ROOT, DRIVE_ROOT, root_gate_path=ROOT_GATE, run_id=RUN_ID)
print(result['status'])
